<a href="https://colab.research.google.com/github/iHakawaTi/RAG-Rest-Agent/blob/main/RAG_Restaurant_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import csv
import json
import datetime
import numpy as np
from google import genai
from google.genai import types


# Configz
MODEL        = "gemini-2.5-flash"      # main conversational brain
EMBED_MODEL  = "gemini-embedding-001"  # Google AI Studio text embedding model
EMBED_DIM    = 768                     # output embedding size (768 / 1536 / 3072)
TOP_K        = 5                       # how many menu items the tool returns
ORDERS_FILE  = "orders.csv"            # local replacement for Google Sheets
RUN_DEMO     = True                    # run a scripted demo before going interactive

# Module-level globals filled in by initialize(). They are used by the menu tool.
CLIENT       = None
MENU_MATRIX  = None
MENU_TEXTS   = None



# The restaurant menu  (the "knowledge base" that gets embedded into the store)
# A small Jordanian / Levantine menu. Prices are in Jordanian Dinar (JOD).

MENU = [
    {"name": "Hummus (حمص)",            "category": "Starters",
     "desc": "Creamy chickpea dip with tahini, lemon and olive oil.",
     "options": {"Small": 1.50, "Large": 2.50}},

    {"name": "Mutabbal (متبل)",          "category": "Starters",
     "desc": "Smoky grilled eggplant blended with tahini and garlic.",
     "options": {"Small": 1.75, "Large": 2.75}},

    {"name": "Fattoush (فتوش)",          "category": "Salads",
     "desc": "Fresh salad with toasted bread, sumac and pomegranate molasses.",
     "options": {"Regular": 2.50}},

    {"name": "Tabbouleh (تبولة)",        "category": "Salads",
     "desc": "Parsley, bulgur, tomato, mint and lemon.",
     "options": {"Regular": 2.50}},

    {"name": "Chicken Shawarma (شاورما دجاج)", "category": "Mains",
     "desc": "Marinated chicken with garlic sauce and pickles in bread.",
     "options": {"Sandwich": 1.50, "Plate": 4.50}},

    {"name": "Meat Shawarma (شاورما لحمة)",    "category": "Mains",
     "desc": "Spiced beef shawarma with tahini sauce.",
     "options": {"Sandwich": 1.75, "Plate": 5.00}},

    {"name": "Mixed Grill (مشاوي مشكلة)", "category": "Mains",
     "desc": "Shish tawook, kofta and lamb chops with grilled vegetables.",
     "options": {"For 1": 7.50, "For 2": 14.00}},

    {"name": "Mansaf (منسف)",            "category": "Mains",
     "desc": "Jordan's national dish: lamb cooked in jameed yogurt over rice.",
     "options": {"Personal": 6.50, "Family": 22.00}},

    {"name": "Cheese Manakish (مناقيش جبنة)", "category": "Pastries",
     "desc": "Oven-baked flatbread topped with melted akkawi cheese.",
     "options": {"Regular": 1.25}},

    {"name": "Zaatar Manakish (مناقيش زعتر)", "category": "Pastries",
     "desc": "Flatbread with thyme, sesame and olive oil.",
     "options": {"Regular": 1.00}},

    {"name": "Kunafa (كنافة)",           "category": "Desserts",
     "desc": "Warm shredded pastry with sweet cheese and syrup.",
     "options": {"Slice": 2.00, "Tray": 12.00}},

    {"name": "Fresh Lemon Mint (ليمون نعناع)", "category": "Drinks",
     "desc": "Blended fresh lemon with mint.",
     "options": {"Glass": 1.50}},

    {"name": "Soft Drink",               "category": "Drinks",
     "desc": "Cola, Sprite or Fanta (330 ml can).",
     "options": {"Can": 0.75}},
]


def format_menu_item(item):
    """Turn one menu dict into a single readable string for embedding/answering."""
    sizes = ", ".join(f"{size}: {price:.2f} JOD"
                      for size, price in item["options"].items())
    return (f"Category: {item['category']}\n"
            f"Item: {item['name']}\n"
            f"Description: {item['desc']}\n"
            f"Sizes/Prices: {sizes}")


# The AI Agent's instructions (adapted from a running system w shortcuts)
SYSTEM_PROMPT = """You are a sales and customer support agent for a restaurant.
You reply to messages on Instagram, Messenger and WhatsApp.

LANGUAGE & TONE
- If the customer writes in Arabic, reply in friendly Jordanian Arabic, like a real human staff member.
- If the customer writes in English, reply in simple, clear English.
- Keep replies short, friendly and non-repetitive. Vary your wording.
- Never say or imply that you are an AI, and never mention tools or technical details.

USING THE MENU TOOL
- For ANY question about prices, item names, availability, ingredients, sizes, or
  categories you MUST call the tool `search_menu` BEFORE you answer.
- Answer ONLY using the text the tool returns. Do NOT invent prices or items.
- Copy item names and prices EXACTLY as they appear in the tool result.
- For greetings or simple non-menu chat you may answer directly without the tool.

RECOMMENDATIONS
- If the customer asks "what do you recommend?", first ask about: number of people,
  food preference (meat / chicken / vegetarian / pastries / drinks), and approximate
  budget. Then suggest 2-3 matching options from the menu.

TAKING ORDERS
- When the customer wants to order, collect, in this order: Name, Address
  (area + nearest landmark), Items and quantities, Sizes/options, Mobile number.

PLACING AN ORDER (VERY IMPORTANT)
- If the customer has clearly provided ALL of: name, full address (area + landmark),
  items with quantities, and phone number, then you MUST do TWO things in your reply:
    1. First, reply to the customer in a friendly way confirming the order.
    2. Then, on a NEW line after your reply, output a JSON block in EXACTLY this format:

ORDER_JSON: {"name": "<customer name>", "address": "<full address and landmark>", "items": "<items and quantities in one line>", "phone": "<phone number>", "channel": "<instagram / whatsapp / messenger>", "contactId": "<contact id>"}

- The line MUST start with: ORDER_JSON:
- The JSON MUST be valid JSON (double quotes, no comments).
- If the order information is incomplete, DO NOT output ORDER_JSON at all.

UNCLEAR QUESTIONS
- If a question is unclear, ask ONE simple clarifying question before answering.
"""


# Vector store (RAG)

def embed(texts, task_type):
    """
    Return an (n x dim) float32 matrix of Gemini embeddings for a list of strings.
    `task_type` should be "RETRIEVAL_DOCUMENT" for stored items and
    "RETRIEVAL_QUERY" for the customer's search query (this is how Google AI Studio
    embeddings are tuned for retrieval / RAG).
    """
    resp = CLIENT.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBED_DIM,
        ),
    )
    return np.array([e.values for e in resp.embeddings], dtype=np.float32)


def _l2_normalise(matrix):
    """L2-normalise each row so cosine similarity == dot product."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9
    return matrix / norms


def build_menu_index():
    """
    Build the in-memory vector store: embed every menu item once and store the
    L2-normalised vectors. This is the local stand-in for the Pinecone 'docx' index.
    """
    global MENU_MATRIX, MENU_TEXTS
    MENU_TEXTS = [format_menu_item(item) for item in MENU]
    vectors = embed(MENU_TEXTS, task_type="RETRIEVAL_DOCUMENT")
    MENU_MATRIX = _l2_normalise(vectors)
    print(f"[setup] Menu vector store ready: {len(MENU_TEXTS)} items embedded "
          f"({EMBED_MODEL}, dim={EMBED_DIM}).")


def search_menu(query: str) -> str:
    """Search the restaurant menu and return the most relevant items.

    Use this for ANY question about prices, item names, availability,
    ingredients, sizes/options, or menu categories. It returns the matching
    menu items with their exact names, descriptions, sizes and prices, which
    must be treated as the only source of truth about the menu.

    Args:
        query: The customer's menu-related question or keywords (English or Arabic).

    Returns:
        A text block with the top matching menu items.
    """
    print(f"   [agent] looking up the menu for: \"{query}\"")
    q = embed([query], task_type="RETRIEVAL_QUERY")[0]
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = MENU_MATRIX @ q                      # cosine similarity for every item
    top_idx = np.argsort(-scores)[:TOP_K]         # indices of the highest scores
    return "\n\n".join(MENU_TEXTS[i] for i in top_idx)


# The AI Agent
def create_chat():
    """
    Create a new Gemini chat session configured as the restaurant agent:
      * system instruction = the agent's rules
      * tools = [search_menu]  (Gemini auto-generates the schema from the function)
      * automatic function calling = Gemini runs the tool and feeds the result back
    The returned chat object also stores conversation history (the agent's memory).
    """
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[search_menu],
        temperature=0.3,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            maximum_remote_calls=5
        ),
    )
    return CLIENT.chats.create(model=MODEL, config=config)


# Order handling
def parse_order(reply_text):
    """
    Split the agent's reply into (customer_text, order_dict).
    `order_dict` is None when no complete order (no ORDER_JSON) was produced.
    Mirrors the original 'Code in JavaScript' parser node.
    """
    marker = "ORDER_JSON:"
    idx = reply_text.find(marker)
    if idx == -1:
        return reply_text.strip(), None

    customer_text = reply_text[:idx].strip()
    json_part = reply_text[idx + len(marker):].strip()
    try:
        order = json.loads(json_part)
    except json.JSONDecodeError:
        order = None          # malformed JSON -> treat as no order, keep the text
    return customer_text, order


def clean_text(text):
    """Collapse newlines into spaces so the reply is a single line."""
    return " ".join(text.split())


def log_order_to_csv(order):
    """Append one confirmed order to a local CSV."""
    file_is_new = not os.path.exists(ORDERS_FILE)
    with open(ORDERS_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if file_is_new:
            writer.writerow(["Timestamp", "Name", "Channel",
                             "ContactId", "Address", "Items", "Phone"])
        writer.writerow([
            datetime.datetime.now().isoformat(timespec="seconds"),
            order.get("name", ""),
            order.get("channel", "instagram"),
            order.get("contactId", ""),
            order.get("address", ""),
            order.get("items", ""),
            order.get("phone", ""),
        ])


# Glue: process a single customer message end-to-end
def process_message(chat, user_text, contact_id="cust_001", first_name="Guest"):
    """
    Wrap one customer message with light metadata,
    send it to the agent, capture any order, and return the clean reply text.
    """
    wrapped = (
        f"Customer message:\n{user_text}\n\n"
        f"Metadata (for personalization only, do NOT answer from this):\n"
        f"- name: {first_name}\n"
        f"- contactId: {contact_id}\n"
        f"- channel: instagram\n"
    )

    response = chat.send_message(wrapped)          # agent thinks + may call the tool
    raw_reply = (response.text or "").strip()

    customer_text, order = parse_order(raw_reply)
    if order is not None:
        log_order_to_csv(order)
        print(f"   [system] Complete order captured -> saved to {ORDERS_FILE}")
        print(f"   [system] {json.dumps(order, ensure_ascii=False)}")

    return clean_text(customer_text)


# Set-up: API key + client + vector store

def get_api_key():
    """Read the Google AI Studio key from the environment"""
    for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        if os.environ.get(var):
            return os.environ[var]
    try:
        from google.colab import userdata
        for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                key = userdata.get(var)
                if key:
                    return key
            except Exception:
                continue
    except Exception:
        pass
    raise RuntimeError(
        "No API key found."
    )


def initialize():
    """Create the Gemini client and build the menu vector store."""
    global CLIENT
    CLIENT = genai.Client(api_key=get_api_key())
    build_menu_index()


# Demo + interactive entry points
def demo():
    """A scripted conversation so the doc can see the full agent flow at once."""
    print("\n" + "=" * 72)
    print(" DEMO CONVERSATION (scripted)")
    print("=" * 72)

    chat = create_chat()
    scripted_turns = [
        "Hi! What do you have for vegetarians and how much?",
        "كم سعر شاورما الدجاج؟",          # Arabic: "How much is the chicken shawarma?"
        # A single message that contains a complete order ->so in theory it should trigger ORDER_JSON:
        ("I'd like to order. Name: Omar Khaled. Address: Sweifieh, next to the "
         "City Mall roundabout. Items: 2 Chicken Shawarma plates and 1 Kunafa tray. "
         "Phone: 0791234567."),
    ]

    for turn in scripted_turns:
        print(f"\nCustomer: {turn}")
        reply = process_message(chat, turn, first_name="Omar")
        print(f"Agent:    {reply}")

    print("=" * 72 + "\n")


def chat_loop():
    """Interactive terminal chat. Type 'quit' to exit."""
    print("Interactive mode. Type your message (or 'quit' to exit).\n")
    chat = create_chat()
    while True:
        try:
            user_text = input("Customer: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break
        if user_text.lower() in {"quit", "exit", "q"}:
            print("Goodbye!")
            break
        if not user_text:
            continue
        reply = process_message(chat, user_text)
        print(f"Agent:    {reply}\n")


if __name__ == "__main__":
    initialize()
    if RUN_DEMO:
        demo()
    chat_loop()

[setup] Menu vector store ready: 13 items embedded (gemini-embedding-001, dim=768).

 DEMO CONVERSATION (scripted)

Customer: Hi! What do you have for vegetarians and how much?
   [agent] looking up the menu for: "vegetarian options"
Agent:    We have a few delicious vegetarian options! * **Mutabbal (متبل)**: Smoky grilled eggplant blended with tahini and garlic. Small: 1.75 JOD, Large: 2.75 JOD. * **Tabbouleh (تبولة)**: Parsley, bulgur, tomato, mint and lemon. Regular: 2.50 JOD. * **Fattoush (فتوش)**: Fresh salad with toasted bread, sumac and pomegranate molasses. Regular: 2.50 JOD.

Customer: كم سعر شاورما الدجاج؟
   [agent] looking up the menu for: "شاورما الدجاج"
Agent:    سعر شاورما الدجاج هو 1.50 دينار للساندويتش و 4.50 دينار للطبق.

Customer: I'd like to order. Name: Omar Khaled. Address: Sweifieh, next to the City Mall roundabout. Items: 2 Chicken Shawarma plates and 1 Kunafa tray. Phone: 0791234567.
   [system] Complete order captured -> saved to orders.csv
   [system] {"name"